#  Exploratory Data Analysis & Storytelling with Data
### SkillsHunger 2026 AI Internship Program — Task 01
---
**Dataset:** Global Superstore Sales Dataset (Kaggle)  
**Domain:** Retail / Business Analytics  
**Objective:** Uncover trends, patterns, and actionable insights from retail sales data to guide business decisions.

##  STEP 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print('✅ All libraries imported successfully!')

##  STEP 2 — Load Dataset
Using Global Superstore Sales Dataset from Kaggle.  
**Link:** https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

In [ ]:
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

print(f'✅ Dataset Loaded Successfully!')
print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head()

##  STEP 3 — Data Cleaning & Preparation

In [ ]:
print('=== DATASET INFO ===')
df.info()
print(f'\n=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found ✅')

In [ ]:
# ── 3.2 Cleaning Steps ──────────────────────────────────────

dups = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'✅ Removed {dups} duplicate rows')

df['Order Date']  = pd.to_datetime(df['Order Date'])
df['Ship Date']   = pd.to_datetime(df['Ship Date'])
print('✅ Converted Order Date and Ship Date to datetime')

# Extract useful time features
df['Order Year']    = df['Order Date'].dt.year
df['Order Month']   = df['Order Date'].dt.month
df['Order Month Name'] = df['Order Date'].dt.strftime('%b')
df['Ship Days']     = (df['Ship Date'] - df['Order Date']).dt.days
print('✅ Extracted Year, Month, Ship Days features')

# Rename columns for clarity
df.rename(columns={
    'Sub-Category': 'Sub_Category',
    'Ship Mode': 'Ship_Mode',
    'Customer Name': 'Customer_Name',
    'Customer ID': 'Customer_ID',
    'Order Date': 'Order_Date',
    'Ship Date': 'Ship_Date',
    'Order ID': 'Order_ID',
    'Postal Code': 'Postal_Code',
    'Product ID': 'Product_ID',
    'Product Name': 'Product_Name'
}, inplace=True)
print('✅ Column names cleaned for clarity')

print(f'\nFinal Dataset Shape: {df.shape}')
df.head(3)

In [ ]:
# ── 3.3 Basic Statistics 
print('=== KEY BUSINESS METRICS ===')
print(f'Total Orders:    {df["Order_ID"].nunique():,}')
print(f'Total Revenue:   ${df["Sales"].sum():,.0f}')
print(f'Total Profit:    ${df["Profit"].sum():,.0f}')
print(f'Avg Discount:    {df["Discount"].mean()*100:.1f}%')
print(f'Avg Ship Days:   {df["Ship Days"].mean():.1f} days')
print(f'Total Customers: {df["Customer_ID"].nunique():,}')
print(f'Total Products:  {df["Product_ID"].nunique():,}')
print(f'Date Range:      {df["Order_Date"].min().date()} → {df["Order_Date"].max().date()}')

##  STEP 4 — Exploratory Data Analysis & Visualizations
### 5+ Chart Types Covered: Bar, Line, Histogram, Heatmap, Pie, Boxplot, Scatter

In [ ]:
#  VIZ 1 — BAR CHART: Sales & Profit by Category
cat_data = df.groupby('Category')[['Sales', 'Profit']].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📊 VIZ 1 — Sales & Profit by Product Category', fontsize=15, fontweight='bold', y=1.01)

colors_sales  = ['#2E86AB', '#A23B72', '#F18F01']
colors_profit = ['#2ecc71' if v > 0 else '#e74c3c' for v in cat_data['Profit']]

bars1 = axes[0].bar(cat_data['Category'], cat_data['Sales'], color=colors_sales,
                    edgecolor='white', width=0.5)
axes[0].set_title('Total Sales by Category')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for bar, val in zip(bars1, cat_data['Sales']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'${val/1000:.0f}K', ha='center', fontweight='bold', fontsize=11)

bars2 = axes[1].bar(cat_data['Category'], cat_data['Profit'], color=colors_profit,
                    edgecolor='white', width=0.5)
axes[1].set_title('Total Profit by Category')
axes[1].set_ylabel('Profit ($)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for bar, val in zip(bars2, cat_data['Profit']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (2000 if val >= 0 else -6000),
                 f'${val/1000:.0f}K', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('viz1_category_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔍 Insight: Technology has highest sales; Furniture has very low profit despite high sales!')

In [ ]:
#  VIZ 2 — LINE GRAPH: Monthly Sales Trend by Year
monthly = df.groupby(['Order Year', 'Order Month'])['Sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle('📈 VIZ 2 — Monthly Sales Trend by Year', fontsize=15, fontweight='bold')

palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
for i, year in enumerate(sorted(monthly['Order Year'].unique())):
    yr_data = monthly[monthly['Order Year'] == year].sort_values('Order Month')
    ax.plot(yr_data['Order Month'], yr_data['Sales'],
            marker='o', linewidth=2.5, markersize=6,
            label=str(year), color=palette[i % len(palette)])

ax.set_xlabel('Month')
ax.set_ylabel('Sales ($)')
ax.set_title('Sales shows strong Q4 spike every year')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.legend(title='Year', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('viz2_monthly_line.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔍 Insight: Sales peak every November–December (Q4 holiday season). Consistent year-over-year growth!')

In [ ]:
# 📊 VIZ 3 — HISTOGRAM: Profit Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📊 VIZ 3 — Profit & Sales Distribution', fontsize=15, fontweight='bold')

axes[0].hist(df['Profit'], bins=60, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(df['Profit'].mean(), color='red', linestyle='--', lw=2,
                label=f'Mean: ${df["Profit"].mean():.1f}')
axes[0].axvline(df['Profit'].median(), color='orange', linestyle='--', lw=2,
                label=f'Median: ${df["Profit"].median():.1f}')
axes[0].set_title('Profit Distribution')
axes[0].set_xlabel('Profit ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(df['Sales'], bins=60, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].axvline(df['Sales'].mean(), color='blue', linestyle='--', lw=2,
                label=f'Mean: ${df["Sales"].mean():.1f}')
axes[1].set_title('Sales Distribution (log-like skew)')
axes[1].set_xlabel('Sales ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz3_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔍 Insight: Many transactions have negative profit — heavy discounting is hurting margins!')

In [ ]:
# VIZ 4 — HEATMAP: Sub-Category Sales vs Region
pivot = df.pivot_table(values='Sales', index='Sub_Category',
                       columns='Region', aggfunc='sum')
pivot = pivot.div(1000).round(1)  # convert to thousands

fig, ax = plt.subplots(figsize=(10, 10))
fig.suptitle('🔥 VIZ 4 — Sub-Category Sales by Region (in $K)', fontsize=15, fontweight='bold')

sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Sales ($K)'},
            annot_kws={'size': 9})
ax.set_title('Phones & Chairs dominate in all regions', fontsize=11)
ax.set_xlabel('Region')
ax.set_ylabel('Sub-Category')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('viz4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔍 Insight: Phones and Chairs are the top-selling sub-categories across all regions!')

In [ ]:
# 🥧 VIZ 5 — PIE CHART: Sales Share by Region + Segment
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('🥧 VIZ 5 — Sales Share by Region & Customer Segment', fontsize=15, fontweight='bold')

region_sales = df.groupby('Region')['Sales'].sum()
colors_r = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
wedges1, texts1, autotexts1 = axes[0].pie(
    region_sales.values, labels=region_sales.index,
    autopct='%1.1f%%', colors=colors_r, startangle=90,
    textprops={'fontsize': 11}, pctdistance=0.8,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts1: at.set_fontweight('bold')
axes[0].set_title('Sales by Region')

seg_sales = df.groupby('Segment')['Sales'].sum()
colors_s = ['#A23B72', '#2E86AB', '#F18F01']
wedges2, texts2, autotexts2 = axes[1].pie(
    seg_sales.values, labels=seg_sales.index,
    autopct='%1.1f%%', colors=colors_s, startangle=90,
    textprops={'fontsize': 11}, pctdistance=0.8,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts2: at.set_fontweight('bold')
axes[1].set_title('Sales by Customer Segment')

plt.tight_layout()
plt.savefig('viz5_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔍 Insight: West region leads sales. Consumer segment dominates with ~50% of total revenue!')

In [ ]:
# 📦 VIZ 6 — BOXPLOT: Profit by Sub-Category
order = df.groupby('Sub_Category')['Profit'].median().sort_values().index

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('📦 VIZ 6 — Profit Distribution by Sub-Category', fontsize=15, fontweight='bold')

palette_box = ['#e74c3c' if df[df['Sub_Category'] == s]['Profit'].median() < 0
               else '#2ecc71' for s in order]
sns.boxplot(data=df, x='Sub_Category', y='Profit', order=order,
            palette=palette_box, ax=ax, linewidth=1.2, fliersize=3)
ax.axhline(0, color='black', linestyle='--', lw=1.5, alpha=0.6)
ax.set_title('Red = negative median profit | Green = positive median profit')
ax.set_xlabel('Sub-Category')
ax.set_ylabel('Profit ($)')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('viz6_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔍 Insight: Tables and Bookcases have NEGATIVE median profit — they are loss-making sub-categories!')

In [ ]:
# 💠 VIZ 7 — SCATTER PLOT: Discount vs Profit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('💠 VIZ 7 — Discount Impact on Profit', fontsize=15, fontweight='bold')

colors_cat = {'Furniture': '#A23B72', 'Office Supplies': '#2E86AB', 'Technology': '#F18F01'}
for cat, grp in df.groupby('Category'):
    axes[0].scatter(grp['Discount'], grp['Profit'],
                    alpha=0.3, s=15, label=cat, color=colors_cat[cat])
axes[0].axhline(0, color='red', linestyle='--', lw=1.5)
axes[0].set_title('Discount vs Profit by Category')
axes[0].set_xlabel('Discount')
axes[0].set_ylabel('Profit ($)')
axes[0].legend(fontsize=9)

# Correlation line
discount_bins = pd.cut(df['Discount'], bins=10)
disc_profit = df.groupby(discount_bins)['Profit'].mean().reset_index()
disc_profit['Discount_mid'] = disc_profit['Discount'].apply(lambda x: x.mid)
axes[1].bar(disc_profit['Discount_mid'], disc_profit['Profit'],
            width=0.03, color=['#e74c3c' if v < 0 else '#2ecc71' for v in disc_profit['Profit']],
            edgecolor='white')
axes[1].axhline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_title('Avg Profit by Discount Level')
axes[1].set_xlabel('Discount Rate')
axes[1].set_ylabel('Average Profit ($)')

plt.tight_layout()
plt.savefig('viz7_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔍 Insight: Discounts above 20% consistently result in LOSSES. Clear negative correlation!')

##  STEP 5 — Correlation Analysis

In [ ]:
# Correlation Heatmap
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Ship Days']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('🔗 Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, square=True, ax=ax,
            annot_kws={'size': 11})
plt.tight_layout()
plt.savefig('viz_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation with Profit:')
print(corr['Profit'].sort_values(ascending=False))

##  STEP 6 — Top Performers & Outlier Analysis

In [ ]:
# Top 10 Most Profitable Products
top_products = df.groupby('Product_Name')['Profit'].sum().nlargest(10).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('🏆 Top & Bottom 10 Products by Profit', fontsize=15, fontweight='bold')

axes[0].barh(top_products['Product_Name'], top_products['Profit'],
             color='#2ecc71', edgecolor='white')
axes[0].set_title('Top 10 Most Profitable Products')
axes[0].set_xlabel('Total Profit ($)')
axes[0].tick_params(axis='y', labelsize=8)

# Bottom 10 (most loss-making)
bottom_products = df.groupby('Product_Name')['Profit'].sum().nsmallest(10).reset_index()
axes[1].barh(bottom_products['Product_Name'], bottom_products['Profit'],
             color='#e74c3c', edgecolor='white')
axes[1].set_title('Top 10 Loss-Making Products')
axes[1].set_xlabel('Total Profit ($)')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('viz_top_products.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# State-level Sales Performance
state_sales = df.groupby('State')[['Sales', 'Profit']].sum().nlargest(10, 'Sales')

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('🗺️ Top 10 States by Sales Performance', fontsize=14, fontweight='bold')

x = range(len(state_sales))
bars1 = ax.bar([i - 0.2 for i in x], state_sales['Sales'], width=0.4,
               color='#3498db', label='Sales', edgecolor='white')
bars2 = ax.bar([i + 0.2 for i in x], state_sales['Profit'], width=0.4,
               color=['#2ecc71' if v > 0 else '#e74c3c' for v in state_sales['Profit']],
               label='Profit', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(state_sales.index, rotation=30)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.legend(fontsize=10)
ax.set_ylabel('Amount ($)')
ax.set_title('California leads in both sales & profit')

plt.tight_layout()
plt.savefig('viz_states.png', dpi=150, bbox_inches='tight')
plt.show()

##  STEP 7 — Summary of Insights

In [ ]:
print('''
╔═══════════════════════════════════════════════════════════════════╗
║          EDA SUMMARY — GLOBAL SUPERSTORE SALES DATASET           ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  🔍 INSIGHT 1: Q4 Seasonality                                    ║
║  Sales spike every November–December across all years.           ║
║  → Business should increase inventory and staff in Q4.           ║
║                                                                   ║
║  🔍 INSIGHT 2: Discount = Profit Killer                           ║
║  Discounts above 20% consistently result in negative profit.     ║
║  → Discount policy needs strict caps (max 15-20%).               ║
║                                                                   ║
║  🔍 INSIGHT 3: Furniture Category is Underperforming             ║
║  Tables and Bookcases have negative median profit.               ║
║  → Review pricing strategy or discontinue loss-making SKUs.      ║
║                                                                   ║
║  🔍 INSIGHT 4: Technology is the Star Category                   ║
║  Highest sales AND best profit margins.                          ║
║  → Invest more in technology product range expansion.            ║
║                                                                   ║
║  🔍 INSIGHT 5: West Region Leads                                  ║
║  West has ~32% of total sales. California alone dominates.       ║
║  → Replicate West's sales strategy in lagging regions.           ║
║                                                                   ║
║  ✅ ACTIONABLE RECOMMENDATION:                                    ║
║  Implement a "Profit First" discount policy: Cap all discounts   ║
║  at 20%, eliminate discounts on Furniture sub-categories, and    ║
║  redirect savings into Technology category expansion.            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
''')